In [ ]:
"""
Skenario 4: NeuMF + Cluster Embedding + TPE (Model Terkuat)
Menggunakan Data: 5 Fitur Audio (Terbaik setelah EDA Distribusi)

Arsitektur IDENTIK dengan Skenario 2:
  - MLP 3 layer: [emb_dim*2 + cluster_dim → 128 → 64 → 32]
  - Cluster embedding disuntikkan HANYA ke MLP input (GMF bersih)
TPE mengoptimasi: emb_dim, cluster_dim, lr, dropout, batch_size
  - TUNE_EPOCHS=10, N_TRIALS=25
  - Final training: EPOCHS=30 dengan best params
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import math
import pickle
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================
# CONFIG
# =====================
# MENGGUNAKAN FILE DATASET 5 FITUR
CLUSTER_PATH = "item_dataset_5f.csv" 
ENCODER_PATH = "data/pkl/item_encoder.pkl"

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 30
TUNE_EPOCHS  = 10
TOP_K        = 10
NUM_NEG      = 99
N_TRIALS     = 25
RANDOM_STATE = 42

print("=" * 60)
print("  Skenario 4: NeuMF Cluster + TPE (5 Fitur Audio)")
print("=" * 60)
print("Device:", DEVICE)

# =====================
# LOAD DATA
# =====================
print("\nMemuat dataset...")
train_df = pd.read_csv("data/train_dataset.csv")
test_df  = pd.read_csv("data/test_dataset.csv")
full_df  = pd.read_csv("data/user_dataset_final.csv")

n_users = full_df['user_id_enc'].max() + 1
n_items = full_df['item_id_enc'].max() + 1
print(f"Users: {n_users}, Items: {n_items}")
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

user_positive_items = (
    train_df[train_df['label'] == 1]
    .groupby('user_id_enc')['item_id_enc']
    .apply(set)
    .to_dict()
)

# =====================
# LOAD CLUSTER
# =====================
print("\nMemuat data cluster (5 Fitur)...")
cluster_df = pd.read_csv(CLUSTER_PATH)

# Karena di 5 fitur nama kolomnya adalah cluster_5f
cluster_df['cluster'] = cluster_df['cluster_5f']

with open(ENCODER_PATH, "rb") as f:
    item_encoder = pickle.load(f)

track_to_enc = dict(zip(
    item_encoder.classes_,
    item_encoder.transform(item_encoder.classes_)
))
cluster_df['item_id'] = cluster_df['track_id'].map(track_to_enc)
cluster_df = cluster_df.dropna(subset=['item_id'])
cluster_df['item_id'] = cluster_df['item_id'].astype(int)

unique_clusters       = sorted(cluster_df['cluster'].unique())
cluster_map           = {c: i for i, c in enumerate(unique_clusters)}
cluster_df['cluster'] = cluster_df['cluster'].map(cluster_map)

item_cluster = dict(zip(cluster_df['item_id'], cluster_df['cluster']))
n_clusters   = len(unique_clusters)
print(f"Jumlah cluster: {n_clusters}, Item ter-cluster: {len(item_cluster)}")

# =====================
# DATASET
# =====================
clusters_arr = np.array([item_cluster.get(i, 0) for i in train_df['item_id_enc'].values])

dataset = TensorDataset(
    torch.tensor(train_df['user_id_enc'].values).long(),
    torch.tensor(train_df['item_id_enc'].values).long(),
    torch.tensor(clusters_arr).long(),
    torch.tensor(train_df['label'].values).float()
)
print(f"Training samples: {len(dataset)}")

# =====================
# MODEL — IDENTIK DENGAN SKENARIO 2
# =====================
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, n_clusters,
                 emb_dim=64, cluster_dim=16, dropout=0.2):
        super().__init__()
        # GMF path — bersih tanpa cluster
        self.user_gmf    = nn.Embedding(n_users, emb_dim)
        self.item_gmf    = nn.Embedding(n_items, emb_dim)
        # MLP path
        self.user_mlp    = nn.Embedding(n_users, emb_dim)
        self.item_mlp    = nn.Embedding(n_items, emb_dim)
        # Cluster embedding — dimensi terpisah, dioptimasi TPE
        self.cluster_emb = nn.Embedding(n_clusters, cluster_dim)

        mlp_input_dim = emb_dim * 2 + cluster_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.Dropout(dropout),
            nn.ReLU()
        )
        self.output = nn.Linear(emb_dim + 32, 1)

    def forward(self, user, item, cluster):
        gmf = self.user_gmf(user) * self.item_gmf(item)
        mlp_in = torch.cat([
            self.user_mlp(user),
            self.item_mlp(item),
            self.cluster_emb(cluster)
        ], dim=-1)
        mlp = self.mlp(mlp_in)
        x   = torch.cat([gmf, mlp], dim=-1)
        return self.output(x).squeeze()

# =====================
# EVALUASI — LOO + 99 NEGATIF
# =====================
@torch.no_grad()
def evaluate(model, seed=RANDOM_STATE):
    model.eval()
    hits, ndcgs = [], []
    rng = np.random.default_rng(seed)

    test_pos = test_df[test_df['label'] == 1].copy()

    for row in tqdm(test_pos.itertuples(index=False), total=len(test_pos),
                    desc="Evaluating", leave=False):
        user      = int(row.user_id_enc)
        true_item = int(row.item_id_enc)
        positives = user_positive_items.get(user, set())

        negatives = set()
        while len(negatives) < NUM_NEG:
            j = int(rng.integers(n_items))
            if j != true_item and j not in positives:
                negatives.add(j)

        items_eval    = list(negatives) + [true_item]
        users_eval    = [user] * len(items_eval)
        clusters_eval = [item_cluster.get(i, 0) for i in items_eval]

        user_t    = torch.tensor(users_eval).long().to(DEVICE)
        item_t    = torch.tensor(items_eval).long().to(DEVICE)
        cluster_t = torch.tensor(clusters_eval).long().to(DEVICE)

        scores    = torch.sigmoid(model(user_t, item_t, cluster_t)).cpu().numpy()
        rank      = np.argsort(scores)[::-1]
        top_items = np.array(items_eval)[rank[:TOP_K]]

        if true_item in top_items:
            r = int(np.where(top_items == true_item)[0][0]) + 1
            hits.append(1)
            ndcgs.append(1.0 / math.log2(r + 1))
        else:
            hits.append(0)
            ndcgs.append(0.0)

    return np.mean(hits), np.mean(ndcgs)

# =====================
# OBJECTIVE FUNCTION — TPE
# =====================
def objective(trial):
    emb_dim     = trial.suggest_categorical("emb_dim",     [32, 64, 128])
    cluster_dim = trial.suggest_categorical("cluster_dim", [8, 16, 32])
    lr          = trial.suggest_float("lr",                1e-4, 1e-2, log=True)
    dropout     = trial.suggest_float("dropout",           0.0, 0.5)
    batch_size  = trial.suggest_categorical("batch_size",  [512, 1024, 2048])

    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_STATE)
    )
    torch.manual_seed(RANDOM_STATE)
    model     = NeuMF(n_users, n_items, n_clusters,
                      emb_dim=emb_dim, cluster_dim=cluster_dim,
                      dropout=dropout).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(TUNE_EPOCHS):
        model.train()
        for u, i, c, l in loader:
            u, i, c, l = u.to(DEVICE), i.to(DEVICE), c.to(DEVICE), l.to(DEVICE)
            pred = model(u, i, c)
            loss = criterion(pred, l)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    hr10, _ = evaluate(model, seed=RANDOM_STATE)
    return hr10

# =====================
# RUN OPTUNA TPE
# =====================
sampler = TPESampler(seed=RANDOM_STATE)
study   = optuna.create_study(
    direction  = "maximize",
    sampler    = sampler,
    study_name = "NCF_Cluster_TPE_Skenario4_5f"
)

print(f"\nStarting Optuna TPE ({N_TRIALS} trials, TUNE_EPOCHS={TUNE_EPOCHS})...")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_trial.params
print("\nBest Trial:")
print(f"  Value (HR@10): {study.best_trial.value:.4f}")
for key, val in best_params.items():
    print(f"    {key}: {val}")

# Simpan semua trial ke CSV
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
study.trials_dataframe().to_csv(
    OUTPUT_DIR / "optuna_trials_skenario4_5f.csv", index=False
)
print(f"Semua trial disimpan: {OUTPUT_DIR / 'optuna_trials_skenario4_5f.csv'}")

# =====================
# FINAL TRAINING — EPOCHS=30
# =====================
torch.manual_seed(RANDOM_STATE)
final_loader = DataLoader(
    dataset, batch_size=best_params["batch_size"], shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)
final_model = NeuMF(
    n_users, n_items, n_clusters,
    emb_dim     = best_params["emb_dim"],
    cluster_dim = best_params["cluster_dim"],
    dropout     = best_params["dropout"]
).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["lr"])

print(f"\nTraining Final Model ({EPOCHS} epochs) dengan best params:")
for key, val in best_params.items():
    print(f"  {key}: {val}")
print()

for epoch in range(EPOCHS):
    final_model.train()
    total_loss = 0
    for u, i, c, l in final_loader:
        u, i, c, l = u.to(DEVICE), i.to(DEVICE), c.to(DEVICE), l.to(DEVICE)
        pred = final_model(u, i, c)
        loss = criterion(pred, l)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f}")

# =====================
# EVALUASI MULTI-SEED
# =====================
EVAL_SEEDS = [42, 123, 456]
print("\nEvaluasi Multi-Seed...")
hr_list, ndcg_list = [], []
for seed in EVAL_SEEDS:
    hr, ndcg = evaluate(final_model, seed=seed)
    hr_list.append(hr)
    ndcg_list.append(ndcg)
    print(f"  Seed {seed:3d} | HR@{TOP_K}={hr:.4f}, NDCG@{TOP_K}={ndcg:.4f}")

mean_hr   = float(np.mean(hr_list))
mean_ndcg = float(np.mean(ndcg_list))
std_hr    = float(np.std(hr_list))
std_ndcg  = float(np.std(ndcg_list))

print(f"\n{'='*60}")
print(f"  HASIL EVALUASI — Skenario 4: NeuMF Cluster + TPE (5 Fitur)")
print(f"{'='*60}")
print(f"  Mean HR@{TOP_K}    : {mean_hr:.4f} ± {std_hr:.4f}")
print(f"  Mean NDCG@{TOP_K}  : {mean_ndcg:.4f} ± {std_ndcg:.4f}")

# =====================
# SIMPAN HASIL (NAMA BERBEDA)
# =====================
result_df = pd.DataFrame([{
    "skenario"     : 4,
    "model"        : "NeuMF Cluster + TPE (5 Fitur)",
    "cluster"      : True,
    "tpe"          : True,
    "HR@10_mean"   : round(mean_hr, 4),
    "HR@10_std"    : round(std_hr, 4),
    "NDCG@10_mean" : round(mean_ndcg, 4),
    "NDCG@10_std"  : round(std_ndcg, 4),
    "seeds"        : str(EVAL_SEEDS),
    "best_params"  : str(best_params),
    "n_trials"     : N_TRIALS,
    "tune_epochs"  : TUNE_EPOCHS,
    "epochs"       : EPOCHS,
    "notes"        : "MLP 3 layer, cluster 5 fitur, TPE optimized"
}])

result_path = OUTPUT_DIR / "hasil_skenario4_cluster_5f_tpe.csv"
result_df.to_csv(result_path, index=False)
print(f"\nHasil disimpan: {result_path}")

Path("models").mkdir(exist_ok=True)
torch.save({
    "model_state_dict" : final_model.state_dict(),
    "best_params"      : best_params,
    "n_users"          : n_users,
    "n_items"          : n_items,
    "n_clusters"       : n_clusters,
    "HR@10_mean"       : mean_hr,
    "NDCG@10_mean"     : mean_ndcg,
}, "models/skenario4_cluster_5f_tpe.pt")
print("Model disimpan: models/skenario4_cluster_5f_tpe.pt")


  Skenario 4: NeuMF Cluster + TPE (5 Fitur Audio)
Device: cuda

Memuat dataset...
Users: 9529, Items: 4144
Train size: 2758490, Test size: 952900

Memuat data cluster (5 Fitur)...
Jumlah cluster: 3, Item ter-cluster: 4144
Training samples: 2758490

Starting Optuna TPE (25 trials, TUNE_EPOCHS=10)...


Best trial: 24. Best value: 0.879631: 100%|██████████| 25/25 [7:41:05<00:00, 1106.60s/it]  



Best Trial:
  Value (HR@10): 0.8796
    emb_dim: 128
    cluster_dim: 32
    lr: 0.00021162092564197587
    dropout: 0.11470937163672609
    batch_size: 512
Semua trial disimpan: outputs\optuna_trials_skenario4_5f.csv

Training Final Model (30 epochs) dengan best params:
  emb_dim: 128
  cluster_dim: 32
  lr: 0.00021162092564197587
  dropout: 0.11470937163672609
  batch_size: 512

Epoch 01/30 | Loss: 1975.6221
Epoch 02/30 | Loss: 1509.3317
Epoch 03/30 | Loss: 1299.2181
Epoch 04/30 | Loss: 1140.1702
Epoch 05/30 | Loss: 1006.1702
Epoch 06/30 | Loss: 888.6098
Epoch 07/30 | Loss: 779.5138
Epoch 08/30 | Loss: 676.1037
Epoch 09/30 | Loss: 577.4284
Epoch 10/30 | Loss: 486.9050
Epoch 11/30 | Loss: 405.5381
Epoch 12/30 | Loss: 334.1302
Epoch 13/30 | Loss: 272.6800
Epoch 14/30 | Loss: 221.2702
Epoch 15/30 | Loss: 179.4192
Epoch 16/30 | Loss: 143.7090
Epoch 17/30 | Loss: 115.3401
Epoch 18/30 | Loss: 93.2727
Epoch 19/30 | Loss: 74.5977
Epoch 20/30 | Loss: 59.9605
Epoch 21/30 | Loss: 48.7214
Epoch

  Seed  42 | HR@10=0.8644, NDCG@10=0.6418


  Seed 123 | HR@10=0.8630, NDCG@10=0.6411


  Seed 456 | HR@10=0.8635, NDCG@10=0.6390

  HASIL EVALUASI — Skenario 4: NeuMF Cluster + TPE (5 Fitur)
  Mean HR@10    : 0.8636 ± 0.0006
  Mean NDCG@10  : 0.6406 ± 0.0012

Hasil disimpan: outputs\hasil_skenario4_cluster_5f_tpe.csv
Model disimpan: models/skenario4_cluster_5f_tpe.pt


In [2]:
"""
Skenario 4: NeuMF + K-Means Cluster + TPE
Menggunakan Data: 5 Fitur Audio

Arsitektur MLP 2 layer:
  - [emb_dim*2 + cluster_dim → 64 → 32]
  - Cluster embedding disuntikkan HANYA ke MLP input
  - GMF tetap bersih tanpa cluster

TPE mengoptimasi:
  - emb_dim
  - cluster_dim
  - lr
  - dropout
  - batch_size

TUNE_EPOCHS = 10
N_TRIALS    = 25
Final training = 30 epoch menggunakan best params
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import math
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

# =====================
# CONFIG
# =====================
CLUSTER_PATH = "../data/item_cluster_4144.csv"

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 30
TUNE_EPOCHS  = 10
N_TRIALS     = 25

TOP_K        = 10
NUM_NEG      = 99
RANDOM_STATE = 42
EVAL_SEEDS   = [42, 123, 456]

print("=" * 60)
print("  Skenario 4: NeuMF + K-Means Cluster + TPE")
print("  5 Fitur Audio | MLP 2 Layer | TPE Optimization")
print("=" * 60)
print("Device:", DEVICE)

# =====================
# LOAD DATA
# =====================
print("\nMemuat dataset...")
train_df = pd.read_csv("../data/train_dataset_cluster_5f.csv")
test_df  = pd.read_csv("../data/test_dataset_cluster_5f.csv")
full_df  = pd.read_csv("../data/user_dataset_final.csv")

n_users = full_df["user_id_enc"].max() + 1
n_items = full_df["item_id_enc"].max() + 1

print(f"Users    : {n_users}")
print(f"Items    : {n_items}")
print(f"Train    : {len(train_df):,}")
print(f"Test     : {len(test_df):,}")

user_positive_items = (
    train_df[train_df["label"] == 1]
    .groupby("user_id_enc")["item_id_enc"]
    .apply(set)
    .to_dict()
)

# =====================
# LOAD CLUSTER
# =====================
print("\nMemuat data cluster...")

cluster_df = pd.read_csv(CLUSTER_PATH)

# Pastikan cluster menjadi index 0 sampai n_clusters-1
unique_clusters = sorted(cluster_df["cluster_5f"].dropna().unique())
cluster_map = {c: idx for idx, c in enumerate(unique_clusters)}

cluster_df["cluster_idx"] = (
    cluster_df["cluster_5f"]
    .map(cluster_map)
    .astype(int)
)

item_cluster = dict(zip(
    cluster_df["item_id"].astype(int),
    cluster_df["cluster_idx"].astype(int)
))

n_clusters = len(unique_clusters)

print(f"Jumlah cluster  : {n_clusters}")
print(f"Item ter-cluster: {len(item_cluster):,}")

# =====================
# DATASET
# =====================
if "cluster_5f" in train_df.columns:
    clusters_arr = (
        train_df["cluster_5f"]
        .map(cluster_map)
        .fillna(0)
        .astype(int)
        .values
    )
else:
    clusters_arr = np.array([
        item_cluster.get(i, 0)
        for i in train_df["item_id_enc"].values
    ])

dataset = TensorDataset(
    torch.tensor(train_df["user_id_enc"].values).long(),
    torch.tensor(train_df["item_id_enc"].values).long(),
    torch.tensor(clusters_arr).long(),
    torch.tensor(train_df["label"].values).float()
)

print(f"Training samples: {len(dataset):,}")

# =====================
# MODEL — MLP 2 LAYER
# =====================
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, n_clusters,
                 emb_dim=64, cluster_dim=16, dropout=0.2):
        super().__init__()

        # GMF path
        self.user_gmf = nn.Embedding(n_users, emb_dim)
        self.item_gmf = nn.Embedding(n_items, emb_dim)

        # MLP path
        self.user_mlp = nn.Embedding(n_users, emb_dim)
        self.item_mlp = nn.Embedding(n_items, emb_dim)

        # Cluster embedding
        self.cluster_emb = nn.Embedding(n_clusters, cluster_dim)

        mlp_input_dim = emb_dim * 2 + cluster_dim

        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 64),
            nn.Dropout(dropout),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.Dropout(dropout),
            nn.ReLU()
        )

        self.output = nn.Linear(emb_dim + 32, 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [
            self.user_gmf,
            self.item_gmf,
            self.user_mlp,
            self.item_mlp,
            self.cluster_emb
        ]:
            nn.init.normal_(emb.weight, std=0.01)

        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, user, item, cluster):
        gmf = self.user_gmf(user) * self.item_gmf(item)

        mlp_in = torch.cat([
            self.user_mlp(user),
            self.item_mlp(item),
            self.cluster_emb(cluster)
        ], dim=-1)

        mlp = self.mlp(mlp_in)

        x = torch.cat([gmf, mlp], dim=-1)
        return self.output(x).squeeze()

# =====================
# EVALUASI — LOO + 99 NEGATIF
# =====================
@torch.no_grad()
def evaluate(model, seed=RANDOM_STATE):
    model.eval()

    hits, ndcgs = [], []
    rng = np.random.default_rng(seed)

    test_pos = test_df[test_df["label"] == 1].copy()

    for row in tqdm(
        test_pos.itertuples(index=False),
        total=len(test_pos),
        desc="Evaluating",
        leave=False
    ):
        user = int(row.user_id_enc)
        true_item = int(row.item_id_enc)
        positives = user_positive_items.get(user, set())

        negatives = set()
        while len(negatives) < NUM_NEG:
            j = int(rng.integers(n_items))
            if j != true_item and j not in positives:
                negatives.add(j)

        items_eval = list(negatives) + [true_item]
        users_eval = [user] * len(items_eval)
        clusters_eval = [item_cluster.get(i, 0) for i in items_eval]

        user_t = torch.tensor(users_eval).long().to(DEVICE)
        item_t = torch.tensor(items_eval).long().to(DEVICE)
        cluster_t = torch.tensor(clusters_eval).long().to(DEVICE)

        scores = torch.sigmoid(
            model(user_t, item_t, cluster_t)
        ).cpu().numpy()

        rank = np.argsort(scores)[::-1]
        top_items = np.array(items_eval)[rank[:TOP_K]]

        if true_item in top_items:
            r = int(np.where(top_items == true_item)[0][0]) + 1
            hits.append(1)
            ndcgs.append(1.0 / math.log2(r + 1))
        else:
            hits.append(0)
            ndcgs.append(0.0)

    return np.mean(hits), np.mean(ndcgs)

# =====================
# OBJECTIVE FUNCTION — TPE
# =====================
def objective(trial):
    emb_dim = trial.suggest_categorical(
        "emb_dim",
        [32, 64, 128]
    )

    cluster_dim = trial.suggest_categorical(
        "cluster_dim",
        [8, 16, 32]
    )

    lr = trial.suggest_float(
        "lr",
        1e-4,
        1e-2,
        log=True
    )

    dropout = trial.suggest_float(
        "dropout",
        0.0,
        0.5
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [512, 1024, 2048]
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_STATE)
    )

    torch.manual_seed(RANDOM_STATE)

    model = NeuMF(
        n_users=n_users,
        n_items=n_items,
        n_clusters=n_clusters,
        emb_dim=emb_dim,
        cluster_dim=cluster_dim,
        dropout=dropout
    ).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(TUNE_EPOCHS):
        model.train()

        for u, i, c, l in loader:
            u = u.to(DEVICE)
            i = i.to(DEVICE)
            c = c.to(DEVICE)
            l = l.to(DEVICE)

            pred = model(u, i, c)
            loss = criterion(pred, l)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    hr10, ndcg10 = evaluate(model, seed=RANDOM_STATE)

    # Mengikuti kode TPE sebelumnya: objective utama HR@10
    return hr10

# =====================
# RUN OPTUNA TPE
# =====================
sampler = TPESampler(seed=RANDOM_STATE)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    study_name="NeuMF_Cluster_TPE_2Layer_5F"
)

print(f"\nStarting Optuna TPE...")
print(f"N_TRIALS    : {N_TRIALS}")
print(f"TUNE_EPOCHS : {TUNE_EPOCHS}")

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

best_params = study.best_trial.params

print("\nBest Trial:")
print(f"  Best HR@10: {study.best_trial.value:.4f}")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# =====================
# SIMPAN TRIAL OPTUNA
# =====================
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

trials_path = OUTPUT_DIR / "optuna_trials_skenario4_cluster_5f_2layer.csv"
study.trials_dataframe().to_csv(trials_path, index=False)

print(f"\nSemua trial Optuna disimpan: {trials_path}")

# =====================
# FINAL TRAINING
# =====================
print("\nTraining Final Model menggunakan best params...")

torch.manual_seed(RANDOM_STATE)

final_loader = DataLoader(
    dataset,
    batch_size=best_params["batch_size"],
    shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)

final_model = NeuMF(
    n_users=n_users,
    n_items=n_items,
    n_clusters=n_clusters,
    emb_dim=best_params["emb_dim"],
    cluster_dim=best_params["cluster_dim"],
    dropout=best_params["dropout"]
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=best_params["lr"]
)

print("\nBest Params:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

print("\nMemulai final training...")

for epoch in range(EPOCHS):
    final_model.train()
    total_loss = 0

    for u, i, c, l in final_loader:
        u = u.to(DEVICE)
        i = i.to(DEVICE)
        c = c.to(DEVICE)
        l = l.to(DEVICE)

        pred = final_model(u, i, c)
        loss = criterion(pred, l)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f}")

# =====================
# EVALUASI MULTI-SEED
# =====================
print("\nEvaluasi Multi-Seed...")

hr_list = []
ndcg_list = []

for seed in EVAL_SEEDS:
    hr, ndcg = evaluate(final_model, seed=seed)

    hr_list.append(hr)
    ndcg_list.append(ndcg)

    print(
        f"  Seed {seed:3d} | "
        f"HR@{TOP_K}={hr:.4f}, "
        f"NDCG@{TOP_K}={ndcg:.4f}"
    )

mean_hr = float(np.mean(hr_list))
mean_ndcg = float(np.mean(ndcg_list))
std_hr = float(np.std(hr_list))
std_ndcg = float(np.std(ndcg_list))

print(f"\n{'=' * 60}")
print("  HASIL EVALUASI — NeuMF + K-Means Cluster + TPE")
print("  MLP 2 Layer [64, 32]")
print(f"{'=' * 60}")
print(f"  Mean HR@{TOP_K}   : {mean_hr:.4f} ± {std_hr:.4f}")
print(f"  Mean NDCG@{TOP_K} : {mean_ndcg:.4f} ± {std_ndcg:.4f}")

# =====================
# SIMPAN HASIL
# =====================
result_df = pd.DataFrame([{
    "skenario": "4",
    "model": "NeuMF + K-Means Cluster + TPE (5 Fitur Audio, MLP 2-Layer)",
    "cluster": True,
    "tpe": True,
    "HR@10_mean": round(mean_hr, 4),
    "HR@10_std": round(std_hr, 4),
    "NDCG@10_mean": round(mean_ndcg, 4),
    "NDCG@10_std": round(std_ndcg, 4),
    "seeds": str(EVAL_SEEDS),
    "best_params": str(best_params),
    "emb_dim": best_params["emb_dim"],
    "cluster_dim": best_params["cluster_dim"],
    "dropout": best_params["dropout"],
    "lr": best_params["lr"],
    "batch_size": best_params["batch_size"],
    "n_clusters": n_clusters,
    "n_trials": N_TRIALS,
    "tune_epochs": TUNE_EPOCHS,
    "epochs": EPOCHS,
    "notes": "MLP 2 layer [64,32], cluster embedding hanya di MLP, TPE optimized"
}])

result_path = OUTPUT_DIR / "hasil_skenario4_cluster_5f_tpe_2layer.csv"
result_df.to_csv(result_path, index=False)

print(f"\nHasil disimpan: {result_path}")

Path("models").mkdir(exist_ok=True)

model_path = "models/skenario4_cluster_5f_tpe_2layer.pt"

torch.save({
    "model_state_dict": final_model.state_dict(),
    "best_params": best_params,
    "n_users": n_users,
    "n_items": n_items,
    "n_clusters": n_clusters,
    "HR@10_mean": mean_hr,
    "HR@10_std": std_hr,
    "NDCG@10_mean": mean_ndcg,
    "NDCG@10_std": std_ndcg,
    "mlp_layer": [64, 32],
    "cluster": True,
    "tpe": True
}, model_path)

print(f"Model disimpan: {model_path}")

  Skenario 4: NeuMF + K-Means Cluster + TPE
  5 Fitur Audio | MLP 2 Layer | TPE Optimization
Device: cuda

Memuat dataset...
Users    : 9529
Items    : 4144
Train    : 2,758,490
Test     : 952,900

Memuat data cluster...
Jumlah cluster  : 3
Item ter-cluster: 4,144
Training samples: 2,758,490

Starting Optuna TPE...
N_TRIALS    : 25
TUNE_EPOCHS : 10


Best trial: 16. Best value: 0.878686: 100%|██████████| 25/25 [4:37:44<00:00, 666.56s/it]  



Best Trial:
  Best HR@10: 0.8787
  emb_dim: 64
  cluster_dim: 16
  lr: 0.0006143217602527595
  dropout: 0.08082374580629714
  batch_size: 2048

Semua trial Optuna disimpan: outputs\optuna_trials_skenario4_cluster_5f_2layer.csv

Training Final Model menggunakan best params...

Best Params:
  emb_dim: 64
  cluster_dim: 16
  lr: 0.0006143217602527595
  dropout: 0.08082374580629714
  batch_size: 2048

Memulai final training...
Epoch 01/30 | Loss: 500.6572
Epoch 02/30 | Loss: 372.3667
Epoch 03/30 | Loss: 322.0820
Epoch 04/30 | Loss: 285.0426
Epoch 05/30 | Loss: 255.2936
Epoch 06/30 | Loss: 230.0130
Epoch 07/30 | Loss: 207.2724
Epoch 08/30 | Loss: 186.1695
Epoch 09/30 | Loss: 166.2982
Epoch 10/30 | Loss: 148.0509
Epoch 11/30 | Loss: 131.3969
Epoch 12/30 | Loss: 116.6137
Epoch 13/30 | Loss: 103.7711
Epoch 14/30 | Loss: 92.1242
Epoch 15/30 | Loss: 82.1275
Epoch 16/30 | Loss: 73.5603
Epoch 17/30 | Loss: 65.8494
Epoch 18/30 | Loss: 59.0787
Epoch 19/30 | Loss: 53.3398
Epoch 20/30 | Loss: 48.0376

  Seed  42 | HR@10=0.8470, NDCG@10=0.6171


  Seed 123 | HR@10=0.8447, NDCG@10=0.6153


  Seed 456 | HR@10=0.8431, NDCG@10=0.6130

  HASIL EVALUASI — NeuMF + K-Means Cluster + TPE
  MLP 2 Layer [64, 32]
  Mean HR@10   : 0.8449 ± 0.0016
  Mean NDCG@10 : 0.6151 ± 0.0017

Hasil disimpan: outputs\hasil_skenario4_cluster_5f_tpe_2layer.csv
Model disimpan: models/skenario4_cluster_5f_tpe_2layer.pt
